# Build the unified structure library

Merges three public natural-product / metabolite databases (DNP, LOTUS, HMDB)
into one deduplicated, RDKit-normalized structure table:
`inchi`, `inchikey`, `smiles`, `formula`, `name`, `organism`, `source_db`.

**This notebook exists for transparency, not necessarily to be re-run.** The
three source files are large (tens of MB to several GB) and are not part of
this repository (see the main [README](../README.md) — raw data lives outside
the repo). If you don't have local copies of DNP/LOTUS/HMDB, read this
notebook to see exactly how the merge works; you don't need to execute it.
If you're bringing your own library instead, use the app's Setup page +
In-silico Library page, which accepts any CSV/Parquet with a structure
column — no need to touch this notebook at all.

**What's deliberately NOT computed here**: `exact_mass` and `has_primary_amine`.
Neither is something any of these three databases actually supplies — they're
properties specific to *this project's* later steps (picking which compounds
to acylate, and computing adduct masses for matching), not general facts about
a compound. Both are cheap to recompute from `inchi` via RDKit and are added
only where they're actually used (`scripts/insilico_library/build_suspect_library.py`
and the In-silico Library GUI page), not stored here. Keeping this table to
only genuinely structural/metadata columns is what makes it a fair example of
what a *user's own* library would look like, too.

In [1]:
import os
import sys
import time

sys.path.insert(0, os.path.abspath("../scripts"))
from insilico_library.db_loader import load_dnp, load_lotus, load_hmdb, merge_rows, compute_primary_amine_flags

# These paths point outside the repo, into the private data folder described
# in the main README's directory layout (`../../data/databases/` relative to
# this notebook) -- adjust them to wherever your own copies live.
DNP_PATH = "../../data/databases/dnp.tsv"
LOTUS_SDF_PATH = "../../data/databases/LOTUS_DB_LATEST/LOTUS_2021_03_simple.sdf"
HMDB_XML_PATH = "../../data/databases/hmdb_metabolites/hmdb_metabolites.xml"

OUTPUT_DIR = "../scripts/insilico_library/data"
OUTPUT_PARQUET = os.path.join(OUTPUT_DIR, "unified_structures.parquet")
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "unified_structures.csv")

## 1. Load each source

Each loader parses one database's own native format and normalizes every row
into the same schema via RDKit (InChI/canonical SMILES/formula are always
recomputed from the structure, never trusted as-is from the source file, so
the result is uniform regardless of which database a row came from). Rows
RDKit can't parse are dropped and counted, not silently skipped.

In [2]:
t0 = time.time()
print(f"Loading DNP from {DNP_PATH} ...")
dnp_rows, dnp_stats = load_dnp(DNP_PATH)
print(f"  done: {dnp_stats}  ({time.time() - t0:.0f}s)")

Loading DNP from ../../data/databases/dnp.tsv ...


  [dnp] 20000 seen, 19999 ok, 20s elapsed, 1012 rec/s


  [dnp] 40000 seen, 39998 ok, 41s elapsed, 986 rec/s


  [dnp] 60000 seen, 59998 ok, 59s elapsed, 1015 rec/s


  [dnp] 80000 seen, 79989 ok, 104s elapsed, 768 rec/s


...

  [dnp] 140000 seen, 139982 ok, 164s elapsed, 852 rec/s


  [dnp] 160000 seen, 159979 ok, 215s elapsed, 745 rec/s


  [dnp] 180000 seen, 179976 ok, 234s elapsed, 770 rec/s


  [dnp] 200000 seen, 199964 ok, 254s elapsed, 789 rec/s


  done: LoadStats(source_db='dnp', n_records_seen=215697, n_parsed_ok=215661, n_parse_failed=36)  (277s)


In [3]:
t0 = time.time()
print(f"Loading LOTUS from {LOTUS_SDF_PATH} ...")
lotus_rows, lotus_stats = load_lotus(LOTUS_SDF_PATH)
print(f"  done: {lotus_stats}  ({time.time() - t0:.0f}s)")

Loading LOTUS from ../../data/databases/LOTUS_DB_LATEST/LOTUS_2021_03_simple.sdf ...


  [lotus] 20000 seen, 20000 ok, 42s elapsed, 477 rec/s


  [lotus] 40000 seen, 40000 ok, 60s elapsed, 666 rec/s


  [lotus] 60000 seen, 60000 ok, 78s elapsed, 765 rec/s


  [lotus] 80000 seen, 80000 ok, 130s elapsed, 614 rec/s


...

  [lotus] 200000 seen, 200000 ok, 309s elapsed, 647 rec/s


  [lotus] 220000 seen, 220000 ok, 328s elapsed, 671 rec/s


  [lotus] 240000 seen, 240000 ok, 376s elapsed, 638 rec/s


  [lotus] 260000 seen, 260000 ok, 402s elapsed, 647 rec/s


  done: LoadStats(source_db='lotus', n_records_seen=276518, n_parsed_ok=276518, n_parse_failed=0)  (417s)


In [4]:
t0 = time.time()
print(f"Loading HMDB from {HMDB_XML_PATH} ...")
hmdb_rows, hmdb_stats = load_hmdb(HMDB_XML_PATH)
print(f"  done: {hmdb_stats}  ({time.time() - t0:.0f}s)")

Loading HMDB from ../../data/databases/hmdb_metabolites/hmdb_metabolites.xml ...


  [hmdb] 20000 seen, 19999 ok, 100s elapsed, 201 rec/s


  [hmdb] 40000 seen, 39999 ok, 223s elapsed, 179 rec/s


  [hmdb] 60000 seen, 59998 ok, 308s elapsed, 195 rec/s


  [hmdb] 80000 seen, 79998 ok, 398s elapsed, 201 rec/s


...

  [hmdb] 140000 seen, 139997 ok, 708s elapsed, 198 rec/s


  [hmdb] 160000 seen, 159977 ok, 812s elapsed, 197 rec/s


  [hmdb] 180000 seen, 179974 ok, 840s elapsed, 214 rec/s


  [hmdb] 200000 seen, 199974 ok, 910s elapsed, 220 rec/s


  done: LoadStats(source_db='hmdb', n_records_seen=217920, n_parsed_ok=217893, n_parse_failed=27)  (940s)


## 2. Merge + dedupe by InChIKey

The same structure found in more than one source is kept once; `source_db`
records every database it appeared in.

In [5]:
t0 = time.time()
merged = merge_rows([dnp_rows, lotus_rows, hmdb_rows])
total_in = len(dnp_rows) + len(lotus_rows) + len(hmdb_rows)
print(f"Merged {total_in} normalized rows -> {len(merged)} unique structures ({time.time() - t0:.0f}s)")
merged.head()

Merged 710072 normalized rows -> 463329 unique structures (3s)


,inchikey,inchi,smiles,formula,name,organism,source_db
0,ZITBJWXLODLDRH-UHFFFAOYSA-N,InChI=1S/C20H22O7/c1-25-17-8-12(3-5-15(17)21)7...,COc1cc(CC2COC(=O)C2(O)Cc2ccc(O)c(OC)c2)ccc1O,C20H22O7,"3-hydroxy-3,4-bis[(4-hydroxy-3-methoxyphenyl)m...","Isol. from Wikstroemia spp., Passerina vulgari...","dnp,hmdb,lotus"
1,LIAGYBNBPQYUKV-UHFFFAOYSA-N,InChI=1S/C26H32O12/c1-34-19-8-13(3-5-17(19)28)...,COc1cc(CC2COC(=O)C2(Cc2ccc(O)c(OC)c2)OC2OC(O)C...,C26H32O12,NaN,Constit. of the roots of Pulsatilla koreana,dnp
2,ZZBYDHWGFRRSDG-UHFFFAOYSA-N,InChI=1S/C27H34O12/c1-34-17-6-4-14(9-19(17)36-...,COc1ccc(CC2COC(=O)C2(O)Cc2ccc(OC)c(OC3OC(CO)C(...,C27H34O12,NaN,Constit. of Saussurea japonica,dnp
3,NPBCPNXLLFVLDR-UHFFFAOYSA-N,InChI=1S/C24H28O8/c1-15(25)32-24(13-17-7-9-20(...,COc1ccc(CC2COC(=O)C2(Cc2ccc(OC)c(OC)c2)OC(C)=O...,C24H28O8,NaN,Constit. of Bupleurum acutifolium,dnp
4,XQAPIFWFLMEZDZ-UHFFFAOYSA-N,InChI=1S/C21H22O7/c1-24-16-5-4-14(9-18(16)25-2...,COc1ccc(CC2(O)C(=O)OCC2Cc2ccc3c(c2)OCO3)cc1OC,C21H22O7,"4-(2h-1,3-benzodioxol-5-ylmethyl)-3-[(3,4-dime...",Constit. of Bupleurum salicifolium,"dnp,lotus"


## 3. Sanity-check stats

Not stored in the output file -- just to see, at merge time, how many
compounds have a primary amine (the property the rest of the pipeline
actually cares about) without baking that flag into the saved table.

In [6]:
n_amine = int(compute_primary_amine_flags(merged["inchi"]).sum())
print(f"Primary-amine-bearing structures: {n_amine} / {len(merged)}")
print()
print("Source DB combination counts:")
print(merged["source_db"].value_counts().to_string())

Primary-amine-bearing structures: 23141 / 463329

Source DB combination counts:
source_db
hmdb              203990
dnp               117109
dnp,lotus          69024
lotus              59308
dnp,hmdb,lotus      7108
dnp,hmdb            4797
hmdb,lotus          1993


## 4. Write the merged table

Written under `scripts/insilico_library/data/` -- input data for the rest of
the pipeline (`build_suspect_library.py`, the In-silico Library GUI page), not
a computed pipeline *result* itself (those live under `output/`).

In [7]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
merged.to_parquet(OUTPUT_PARQUET, index=False)
merged.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(merged)} rows -> {OUTPUT_PARQUET}")
print(f"Wrote {len(merged)} rows -> {OUTPUT_CSV}")

Wrote 463329 rows -> ../scripts/insilico_library/data\unified_structures.parquet
Wrote 463329 rows -> ../scripts/insilico_library/data\unified_structures.csv
